### تحميل وفحص بيانات الاختبار
بناءً على طلبك، سنقوم الآن بتحميل ملف بيانات الاختبار وعرض حجمه.

### التحضير لتدريب نموذج AraBERT

نظرًا لأن النماذج التقليدية لم تحقق الدقة المطلوبة، سننتقل الآن إلى استخدام نموذج لغوي كبير ومعالج مسبقًا (Pre-trained Language Model) لتحليل المشاعر. سنستخدم **AraBERT**، وهو نسخة من نموذج BERT تم تدريبها خصيصًا على اللغة العربية، مما يجعله فعالًا للغاية في فهم سياق النصوص العربية.

تتطلب هذه العملية تثبيت مكتبات إضافية وتجهيز البيانات بشكل مختلف ليناسب متطلبات نماذج Transformers.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset, DatasetDict
import matplotlib.pyplot as plt
import seaborn as sns
import evaluate # لتسهيل حساب المقاييس

### تجهيز البيانات لنموذج AraBERT

نحتاج إلى تحويل بياناتنا النصية والتسميات لتكون متوافقة مع نموذج AraBERT. يتضمن ذلك ترميز النصوص باستخدام `AutoTokenizer` الخاص بـ AraBERT وتحويل التسميات إلى أرقام صحيحة.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer
from datasets import Dataset
import re
import string
import nltk
from nltk.corpus import stopwords

# تحديد مسارات ملفات بيانات التدريب (هذه يجب أن تكون معرفة من خلايا سابقة أو يتم تحميلها هنا)
file_path_ss2030 = '/content/Arabic Sentiment Analysis Dataset - SS2030.csv'
file_path_training_1 = '/content/training_data (1).csv'
file_path_training = '/content/training_data.csv'

# تحميل كل ملف إلى DataFrame
df_ss2030 = pd.read_csv(file_path_ss2030)
df_training_1 = pd.read_csv(file_path_training_1)
df_training = pd.read_csv(file_path_training)

# إعادة تسمية الأعمدة في df_ss2030 لتتوافق مع DataFrames الأخرى
df_ss2030 = df_ss2030.rename(columns={'text': 'tweet', 'Sentiment': 'label'})

# إعادة تسمية عمود 'sentiment' إلى 'label' في df_training_1 و df_training
df_training_1 = df_training_1.rename(columns={'sentiment': 'label'})
df_training = df_training.rename(columns={'sentiment': 'label'})

# تحديد الأعمدة الأساسية التي نريد دمجها
columns_to_keep = ['tweet', 'label']

# اختيار الأعمدة المطلوبة من كل DataFrame قبل الدمج
df_ss2030_processed = df_ss2030[columns_to_keep]
df_training_1_processed = df_training_1[columns_to_keep]
df_training_processed = df_training[columns_to_keep]

# دمج جميع DataFrames الخاصة بالتدريب
df_combined = pd.concat([df_ss2030_processed, df_training_1_processed, df_training_processed], ignore_index=True)

# إزالة الصفوف المكررة
df_combined.drop_duplicates(inplace=True)

# إزالة الصفوف التي تحتوي على قيم مفقودة في أي من العمودين 'tweet' أو 'label'.
df_combined.dropna(subset=['tweet', 'label'], inplace=True)

# توحيد تصنيفات المشاعر
def standardize_label(label):
    label_str = str(label).strip()
    if label_str in ['1', 'POS']:
        return 'Positive'
    elif label_str in ['0', 'NEG']:
        return 'Negative'
    elif label_str == 'NEU':
        return 'Neutral'
    else:
        return label_str

df_combined['label'] = df_combined['label'].apply(standardize_label)

# التأكد من تحميل الكلمات الوقفية العربية
try:
    stopwords.words('arabic')
except LookupError:
    nltk.download('stopwords')

# قائمة بالرموز التعبيرية الشائعة
emoji_pattern = re.compile("[" # Start of pattern
                           "\U0001F600-\U0001F64F"  # emoticons
                           "\U0001F300-\U0001F5FF"  # symbols & pictographs
                           "\U0001F680-\U0001F6FF"  # transport & map symbols
                           "\U0001F1E0-\U0001F1FF"  # flags (iOS)
                           "\U00002702-\U000027B0"  # Dingbats
                           "\U000024C2-\U0001F251"
                           "]+", flags=re.UNICODE)

def clean_arabic_text(text):
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+|#\w+', '', text)
    text = emoji_pattern.sub(r'', text)
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub(r'\d+', '', text) # إزالة الأرقام
    text = re.sub(r'[\u064b-\u0652]', '', text) # تنوين وشدة وفتحة وضمة وكسرة وسكون
    text = re.sub(r'(.)\1+', r'\1', text)
    text = re.sub(r'[أإآ]', 'ا', text)
    text = re.sub(r'ى', 'ي', text)
    text = re.sub(r'ة', 'ه', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_combined['cleaned_tweet'] = df_combined['tweet'].apply(clean_arabic_text)

# قائمة الكلمات الوقفية العربية
arabic_stopwords = set(stopwords.words('arabic'))

def tokenize_and_remove_stopwords(text):
    tokens = text.split()
    filtered_tokens = [word for word in tokens if word not in arabic_stopwords]
    return ' '.join(filtered_tokens)

df_combined['tokenized_tweet'] = df_combined['cleaned_tweet'].apply(tokenize_and_remove_stopwords)

# تحديد اسم نموذج AraBERT الذي سنستخدمه
model_name = 'aubmindlab/bert-base-arabertv02'

# تحميل المُجَزِّئ (Tokenizer) الخاص بنموذج AraBERT
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Re-initialize LabelEncoder and encode labels for BERT within this cell
# This ensures these variables are defined in this execution context
label_encoder = LabelEncoder()
y_encoded_for_bert = label_encoder.fit_transform(df_combined['label'])

# إضافة التسميات المشفرة إلى DataFrame
df_combined['encoded_label'] = y_encoded_for_bert

# إنشاء تعيين من التسميات الرقمية إلى النصية وبالعكس
label_map = {idx: label for idx, label in enumerate(label_encoder.classes_)}
num_labels = len(label_encoder.classes_)

# تقسيم البيانات إلى مجموعات تدريب واختبار (على النصوص الأصلية)
X_train_text, X_test_text, y_train_bert, y_test_bert = train_test_split(
    df_combined['cleaned_tweet'], df_combined['encoded_label'],
    test_size=0.20, random_state=42, stratify=df_combined['encoded_label']
)

# تحويل البيانات إلى تنسيق Dataset
train_df = pd.DataFrame({'text': X_train_text, 'label': y_train_bert})
test_df = pd.DataFrame({'text': X_test_text, 'label': y_test_bert})

# تحويل Pandas DataFrames إلى Hugging Face Datasets
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

# دالة لترميز النصوص
def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True)

# تطبيق الترميز على مجموعات البيانات
tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True)

# إزالة الأعمدة غير الضرورية وإعادة تسمية عمود التسمية ليتوافق مع Trainer
tokenized_train_dataset = tokenized_train_dataset.remove_columns(['text'])
tokenized_test_dataset = tokenized_test_dataset.remove_columns(['text'])

print('أول مثال من مجموعة التدريب بعد الترميز:')
print(tokenized_train_dataset[0])
print(f'حجم مجموعة التدريب بعد الترميز: {len(tokenized_train_dataset)}')
print(f'حجم مجموعة الاختبار بعد الترميز: {len(tokenized_test_dataset)}')


Map:   0%|          | 0/13380 [00:00<?, ? examples/s]

Map:   0%|          | 0/3346 [00:00<?, ? examples/s]

أول مثال من مجموعة التدريب بعد الترميز:
{'label': 1, 'input_ids': [2, 47476, 634, 487, 45486, 307, 13685, 18799, 195, 2735, 4702, 132, 4294, 15155, 411, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

### تحميل نموذج AraBERT وتحديد مقاييس التقييم

سنقوم الآن بتحميل نموذج AraBERT الجاهز للمهمة (تصنيف تسلسل) وتحديد دالة لحساب مقاييس الأداء مثل الدقة (Accuracy)، الاستدعاء (Recall)، والدقة (Precision) و F1-score.

In [ ]:
from transformers import AutoModelForSequenceClassification
import numpy as np # Ensure numpy is imported if used here
import evaluate # Ensure evaluate is imported if used here

model_bert = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

# ربط التسميات بالنموذج (اختياري لكن جيد للتقارير)
model_bert.config.id2label = label_map
model_bert.config.label2id = {label: idx for idx, label in label_map.items()}

# تحميل مقياس التقييم
metric = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Conside

In [ ]:
from transformers import TrainingArguments # Technically already imported in a previous cell, but adding explicitly as requested.

In [ ]:
from transformers import TrainingArguments

### تعريف وسائط التدريب (Training Arguments) وتدريب نموذج AraBERT

سنحدد الآن المعاملات التي ستتحكم في عملية التدريب، مثل عدد الحقبات (epochs)، حجم الدفعة (batch size)، ومعدل التعلم (learning rate). بعد ذلك، سنقوم بإنشاء `Trainer` وبدء عملية تدريب نموذج AraBERT.

In [ ]:
from transformers import TrainingArguments, Trainer

# تعريف وسائط التدريب
training_args = TrainingArguments(
    output_dir='./results_arabert',
    eval_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3, # يمكن زيادة عدد الحقبات لمزيد من الدقة ولكنها تزيد وقت التدريب
    weight_decay=0.01,
    logging_dir='./logs_arabert',
    logging_steps=500,
    report_to='none' # لا نرسل التقارير إلى أي مكان خارجي
)

# تهيئة Trainer
trainer = Trainer(
    model=model_bert,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_test_dataset,
    compute_metrics=compute_metrics,
)

# تدريب النموذج
trainer.train()

print('تم تدريب نموذج AraBERT بنجاح.')

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


NameError: name 'model_bert' is not defined

### تقييم أداء نموذج AraBERT

بعد التدريب، سنقوم بتقييم أداء نموذج AraBERT على مجموعة الاختبار باستخدام مقاييس شاملة مثل الدقة (Accuracy)، تقرير التصنيف (Classification Report) ومصفوفة الارتباك (Confusion Matrix) لمعرفة مدى تحسن الأداء.

In [ ]:
# تقييم النموذج
eval_results = trainer.evaluate()
print(f'نتائج تقييم نموذج AraBERT: {eval_results}')

# الحصول على التنبؤات على مجموعة الاختبار
predictions_output = trainer.predict(tokenized_test_dataset)
y_pred_bert_encoded = np.argmax(predictions_output.predictions, axis=-1)
y_true_bert_encoded = predictions_output.label_ids

# تحويل التنبؤات والتسميات الحقيقية إلى نصية للعرض
y_pred_bert = label_encoder.inverse_transform(y_pred_bert_encoded)
y_true_bert = label_encoder.inverse_transform(y_true_bert_encoded)

# حساب الدقة
accuracy_bert = accuracy_score(y_true_bert, y_pred_bert)
print(f'دقة نموذج AraBERT (Accuracy): {accuracy_bert:.4f}')

# التحقق من شرط الدقة
if accuracy_bert > 0.85:
    print('تهانينا! دقة نموذج AraBERT تتجاوز 85% المطلوبة.')
else:
    print('ملاحظة: دقة نموذج AraBERT لم تتجاوز 85% بعد. قد تحتاج إلى ضبط المعاملات أو زيادة بيانات التدريب.')

# عرض تقرير التصنيف
print('\nتقرير التصنيف لنموذج AraBERT:\n')
print(classification_report(y_true_bert, y_pred_bert, target_names=label_encoder.classes_))

# عرض مصفوفة الارتباك
conf_matrix_bert = confusion_matrix(y_true_bert, y_pred_bert)
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix_bert, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title('مصفوفة الارتباك لنموذج AraBERT')
plt.xlabel('التصنيف المتوقع')
plt.ylabel('التصنيف الفعلي')
plt.show()

NameError: name 'trainer' is not defined

### تحضير بيانات الاختبار للحفظ

قبل حفظ بيانات الاختبار، من الضروري تطبيق نفس خطوات المعالجة المسبقة التي طبقناها على بيانات التدريب، بما في ذلك تنظيف النصوص وتوحيد تسميات المشاعر. هذا يضمن أن تكون بيانات الاختبار جاهزة للاستخدام بنفس التنسيق.

In [ ]:
# عرض الأعمدة في df_testing قبل المعالجة للتأكد من الأسماء
print('الأعمدة في df_testing:', df_testing.columns.tolist())

# افتراض أن عمود النص هو 'tweet' وعمود المشاعر هو 'sentiment' أو 'label'
# إذا كان عمود المشاعر اسمه 'sentiment'، نقوم بإعادة تسميته إلى 'label'
if 'sentiment' in df_testing.columns and 'label' not in df_testing.columns:
    df_testing = df_testing.rename(columns={'sentiment': 'label'})

# تطبيق دالة التنظيف على عمود النص في df_testing
df_testing['cleaned_tweet'] = df_testing['tweet'].apply(clean_arabic_text)

# تطبيق دالة توحيد التسميات على عمود المشاعر في df_testing (إذا كان موجوداً)
if 'label' in df_testing.columns:
    df_testing['label'] = df_testing['label'].apply(standardize_label)

print('\nأول 5 صفوف من بيانات الاختبار بعد المعالجة:')
display(df_testing[['tweet', 'cleaned_tweet', 'label']].head())

الأعمدة في df_testing: ['tweet', 'sarcasm', 'sentiment', 'dialect']

أول 5 صفوف من بيانات الاختبار بعد المعالجة:


,tweet,cleaned_tweet,label
0,اخوي حانق يالغلا وشفيك معصب؟ عادي تراهم بشر يف...,اخوي حانق يالغلا وشفيك معصب؟ عادي تراهم بشر يف...,Negative
1,اف مو متعوده عليهم سته https://t.co/8igFPx1i26,اف مو متعوده عليهم سته,Negative
2,اللهم اشفِ مرضانا ومرضى المسلمين . . ♥️,الهم اشف مرضانا ومرضي المسلمين,Positive
3,ابشركم طلقت السات 😘.,ابشركم طلقت السات,Positive
4,مؤشر خطير: ٩٠٪ من الشخصيات البرلمانية في الكوي...,مؤشر خطير ٪ من الشخصيات البرلمانيه في الكويت ت...,Negative


### حفظ البيانات المعالجة

الآن وبعد تنظيف وتجهيز كل من بيانات التدريب والاختبار، سنقوم بحفظها في ملفات CSV منفصلة. هذا يسمح بإعادة استخدام هذه البيانات المعالجة لاحقاً دون الحاجة إلى إعادة خطوات المعالجة المسبقة.

In [ ]:
# حفظ بيانات التدريب المعالجة
df_combined[['cleaned_tweet', 'label']].to_csv('processed_training_data.csv', index=False)
print('تم حفظ بيانات التدريب المعالجة في: processed_training_data.csv')

# --- بدء معالجة بيانات الاختبار لضمان توفر الأعمدة المطلوبة ---
# افتراض أن df_testing تم تحميله من خلية سابقة وأن دالتي clean_arabic_text و standardize_label معرفتان.
# إذا كان عمود المشاعر اسمه 'sentiment' في df_testing، نقوم بإعادة تسميته إلى 'label'
if 'sentiment' in df_testing.columns and 'label' not in df_testing.columns:
    df_testing = df_testing.rename(columns={'sentiment': 'label'})

# تطبيق دالة التنظيف على عمود النص في df_testing
df_testing['cleaned_tweet'] = df_testing['tweet'].apply(clean_arabic_text)

# تطبيق دالة توحيد التسميات على عمود المشاعر في df_testing (إذا كان موجوداً)
if 'label' in df_testing.columns:
    df_testing['label'] = df_testing['label'].apply(standardize_label)
# --- انتهاء معالجة بيانات الاختبار ---

# حفظ بيانات الاختبار المعالجة
# نتأكد من وجود عمود 'label' قبل محاولة حفظه في بيانات الاختبار
if 'label' in df_testing.columns:
    df_testing[['cleaned_tweet', 'label']].to_csv('processed_testing_data.csv', index=False)
    print('تم حفظ بيانات الاختبار المعالجة في: processed_testing_data.csv')
else:
    # إذا لم يكن هناك عمود تسمية، نحفظ عمود النص النظيف فقط
    df_testing[['cleaned_tweet']].to_csv('processed_testing_data.csv', index=False)
    print('تم حفظ بيانات الاختبار المعالجة (بدون تسميات) في: processed_testing_data.csv')


تم حفظ بيانات التدريب المعالجة في: processed_training_data.csv
تم حفظ بيانات الاختبار المعالجة في: processed_testing_data.csv


In [ ]:
import pandas as pd

# تحديد مسار ملف بيانات الاختبار
file_path_testing = '/content/testing_data.csv'

# تحميل ملف الاختبار إلى DataFrame
df_testing = pd.read_csv(file_path_testing)

# عرض أول 5 صفوف من بيانات الاختبار
print('أول 5 صفوف من بيانات الاختبار:')
display(df_testing.head())

# عرض معلومات عامة عن بيانات الاختبار (عدد الصفوف، الأعمدة، أنواع البيانات)
print('\nمعلومات حول بيانات الاختبار:')
df_testing.info()

# عرض حجم بيانات الاختبار (عدد الصفوف والأعمدة)
print(f'\nعدد الصفوف في بيانات الاختبار: {df_testing.shape[0]}')
print(f'عدد الأعمدة في بيانات الاختبار: {df_testing.shape[1]}')

أول 5 صفوف من بيانات الاختبار:


,tweet,sarcasm,sentiment,dialect
0,اخوي حانق يالغلا وشفيك معصب؟ عادي تراهم بشر يف...,False,NEG,msa
1,اف مو متعوده عليهم سته https://t.co/8igFPx1i26,True,NEG,msa
2,اللهم اشفِ مرضانا ومرضى المسلمين . . ♥️,False,POS,msa
3,ابشركم طلقت السات 😘.,False,POS,gulf
4,مؤشر خطير: ٩٠٪ من الشخصيات البرلمانية في الكوي...,True,NEG,msa



معلومات حول بيانات الاختبار:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   tweet      3000 non-null   object
 1   sarcasm    3000 non-null   bool  
 2   sentiment  3000 non-null   object
 3   dialect    3000 non-null   object
dtypes: bool(1), object(3)
memory usage: 73.4+ KB

عدد الصفوف في بيانات الاختبار: 3000
عدد الأعمدة في بيانات الاختبار: 4


In [ ]:
import pandas as pd
import re
import string
import nltk
from nltk.corpus import stopwords
from sklearn.preprocessing import LabelEncoder

# تحديد مسارات ملفات بيانات التدريب
file_path_ss2030 = '/content/Arabic Sentiment Analysis Dataset - SS2030.csv'
file_path_training_1 = '/content/training_data (1).csv'
file_path_training = '/content/training_data.csv'

# تحميل كل ملف إلى DataFrame
df_ss2030 = pd.read_csv(file_path_ss2030)
df_training_1 = pd.read_csv(file_path_training_1)
df_training = pd.read_csv(file_path_training)

# إعادة تسمية الأعمدة في df_ss2030 لتتوافق مع DataFrames الأخرى
df_ss2030 = df_ss2030.rename(columns={'text': 'tweet', 'Sentiment': 'label'})

# إعادة تسمية عمود 'sentiment' إلى 'label' في df_training_1 و df_training
df_training_1 = df_training_1.rename(columns={'sentiment': 'label'})
df_training = df_training.rename(columns={'sentiment': 'label'})

# تحديد الأعمدة الأساسية التي نريد دمجها
columns_to_keep = ['tweet', 'label']

# اختيار الأعمدة المطلوبة من كل DataFrame قبل الدمج
df_ss2030_processed = df_ss2030[columns_to_keep]
df_training_1_processed = df_training_1[columns_to_keep]
df_training_processed = df_training[columns_to_keep]

# دمج جميع DataFrames الخاصة بالتدريب
df_combined = pd.concat([df_ss2030_processed, df_training_1_processed, df_training_processed], ignore_index=True)

# إزالة الصفوف المكررة
df_combined.drop_duplicates(inplace=True)

# إزالة الصفوف التي تحتوي على قيم مفقودة في أي من العمودين 'tweet' أو 'label'.
df_combined.dropna(subset=['tweet', 'label'], inplace=True)

# توحيد تصنيفات المشاعر
def standardize_label(label):
    label_str = str(label).strip()
    if label_str in ['1', 'POS']:
        return 'Positive'
    elif label_str in ['0', 'NEG']:
        return 'Negative'
    elif label_str == 'NEU':
        return 'Neutral'
    else:
        return label_str

df_combined['label'] = df_combined['label'].apply(standardize_label)

# التأكد من تحميل الكلمات الوقفية العربية
try:
    stopwords.words('arabic')
except LookupError:
    nltk.download('stopwords')

# قائمة بالرموز التعبيرية الشائعة
emoji_pattern = re.compile("[" # Start of pattern
                           "\U0001F600-\U0001F64F"  # emoticons
                           "\U0001F300-\U0001F5FF"  # symbols & pictographs
                           "\U0001F680-\U0001F6FF"  # transport & map symbols
                           "\U0001F1E0-\U0001F1FF"  # flags (iOS)
                           "\U00002702-\U000027B0"  # Dingbats
                           "\U000024C2-\U0001F251"
                           "]+", flags=re.UNICODE)

def clean_arabic_text(text):
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+|#\w+', '', text)
    text = emoji_pattern.sub(r'', text)
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[\u064b-\u0652]', '', text)
    text = re.sub(r'(.)\1+', r'\1', text)
    text = re.sub(r'[أإآ]', 'ا', text)
    text = re.sub(r'ى', 'ي', text)
    text = re.sub(r'ة', 'ه', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_combined['cleaned_tweet'] = df_combined['tweet'].apply(clean_arabic_text)

# قائمة الكلمات الوقفية العربية
arabic_stopwords = set(stopwords.words('arabic'))

def tokenize_and_remove_stopwords(text):
    tokens = text.split()
    filtered_tokens = [word for word in tokens if word not in arabic_stopwords]
    return ' '.join(filtered_tokens)

df_combined['tokenized_tweet'] = df_combined['cleaned_tweet'].apply(tokenize_and_remove_stopwords)

# تهيئة LabelEncoder لـ AraBERT
label_encoder = LabelEncoder()
y_encoded_for_bert = label_encoder.fit_transform(df_combined['label'])